In [ ]:
from typing import Any, Callable, Dict, List, Union, Tuple, Optional, Set
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import shutil

# Setup

In [ ]:
# Constants
BASE_PATH = os.path.join("D:\\", "Workspaces", "vscode-workspace", "ai_x_medicine", "data")
WATER_DATASET_FILENAME = "env_water.csv"
EARTH_DATASET_FILENAME = "env_earth.csv"
FIRE_DATASET_FILENAME = "env_fire.csv"
WOOD_DATASET_FILENAME = "env_wood.csv"
METAL_DATASET_FILENAME = "env_metal.csv"

FIRE_OUTPUT_PATH = os.path.join(BASE_PATH, "env_fire-new.csv")
OUTPUT_PATH = os.path.join(BASE_PATH, "out", "economical-pain.csv")

def env_data_in(element: str) -> str:
    return os.path.join(BASE_PATH, f"env_{element}.csv")

def env_pain_out(element: str) -> str:
    return os.path.join(BASE_PATH, "out", "env_pains", f"env-{element}-pain.csv")

In [ ]:
# transformation functions
def transform_dictionary(extracted_data: Dict[str, Any], convert: Callable[[Any], float]):
    """
    extracted data: needs to map from Country name to Any data
    convert: converts Any data into a pain value between 0 and 1
    returns: DataFrame with Country and Pain-Value columns
    """
    df_res = []
    for key, value in extracted_data.items():
        df_res.append({
            "Country": key,
            "value": convert(value)
        })
    return pd.DataFrame(df_res)

In [ ]:
# Util functions
def is_valid_data(val):
    return pd.notnull(val) and len(str(val).strip()) > 0

def bar_plot(dataframe: pd.DataFrame):
    plt.figure(figsize=(12, 6))
    dataframe.plot(kind='bar')
    plt.xlabel('Countries')
    plt.ylabel('values')
    # plt.title('')
    plt.tight_layout()
    plt.show()

def nf_linear(val, max_val):
    return val / max_val

def nf_log(val, max_val):
    return np.log(val / max_val)

def nf_log2(val, max_val):
    # todo: fix this metric!
    if val < 1: return 0    # don't allow negatives! but this should only happen vor val == 0 anyways
    return np.log(val) / np.log(max_val)

def normalize_data(dataframe: pd.DataFrame, norm_func: Callable[[float, float], float]):
    max_val = dataframe["value"].max()
    dataframe['value'] = dataframe["value"].apply(lambda x: norm_func(x, max_val))


# Transform data

In [ ]:
# [x] transform Earth data
# Input:    Red List Index in [0, 1] for different years
# Output:   (1 - Red List Index) from the most recent year per country
# see: https://en.wikipedia.org/wiki/Red_List_Index#/media/File:Red_List_Index,_OWID.svg

# (1) load original data
og_earth_data = pd.read_csv(env_data_in("earth"))

# (2) extract the required data
res_earth: Dict[str, Tuple[int, float]] = {}
for _, row in og_earth_data.iterrows():
    country = row["Entity"]
    year = row["Year"]
    value = row["Red List Index"]
    
    if country in res_earth:
        cur_year, cur_val = res_earth[country]
        if cur_year < year:
            res_earth[country] = year, value
    else:
        res_earth[country] = year, value

# (3) transform the extracted data
df_earth = transform_dictionary(res_earth, lambda data: np.round(1-data[1], 3))
df_earth.to_csv(env_pain_out("earth"), index=False)

In [ ]:
# [x] transform fire data (2024)
# Input:    Annual area burnt by wildfires for different years
# Output:   relative (log ratio) Annual area burnt by wildfires for 2024

# (1) load original data
#df_fire = pd.read_csv(os.path.join(BASE_PATH, "elements_standardised", "env-fire-pain-standardised.csv"))
og_fire_data = pd.read_csv(env_data_in("fire"))

# (2) extract the required data
res_fire: Dict[str, float] = {}
for _, row in og_fire_data.iterrows():
    country = row["Entity"]
    year = row["Year"]
    value = row["Annual area burnt by wildfires"]
    
    if year == 2024 and not "world" in country.lower():
        res_fire[country] = value

# (3) transform the extracted data
max_value = max(res_fire.values())
df_fire = transform_dictionary(res_fire, lambda data: nf_log2(data, max_value))
df_fire.to_csv(env_pain_out("fire"), index=False)

df_fire.plot(kind="bar")

In [ ]:
# [x] transform water data
# Input:    Floods with year and Total Affected (humans?)
# Output:   relative (log ratio) Total Affected sum of all floods between 2019 and 2025

# (1) load original data
og_water_data = pd.read_csv(env_data_in("water"))

# (2) extract the required data
res_water: Dict[str, float] = {}
for _, row in og_water_data.iterrows():
    country = row["Country"]
    year = row["Start Year"]
    value = row["Total Affected"]
    if 2019 < year <= 2025:
        if np.isnan(value):
            value = 0
        if country not in res_water:
            res_water[country] = 0
        res_water[country] += value

# (3) transform the extracted data
max_value = max(res_water.values())
df_water = transform_dictionary(res_water, lambda data: nf_log2(data, max_value))
df_water.to_csv(env_pain_out("water"), index=False)

df_water.plot(kind="bar")

In [ ]:
# [x] transform wood data
# Input:    Deforestation in various years (1990, 2000, 2010, 2015)
# Output:   relative (log ratio) Deforestaion in 2015

# (1) load original data
og_wood_data = pd.read_csv(env_data_in("wood"))

# (2) extract the required data
res_wood: Dict[str, float] = {}
for _, row in og_wood_data.iterrows():
    country = row["Entity"]
    year = row["Year"]
    value = row["Deforestation"]
    if 2015 <= year <= 2020:
        if country not in res_wood:
            res_wood[country] = 0
        res_wood[country] += value

# (3) transform the extracted data
max_value = max(res_wood.values())
df_wood = transform_dictionary(res_wood, lambda data: nf_log2(data, max_value))
df_wood.to_csv(env_pain_out("wood"), index=False)

df_wood.plot(kind="bar")

In [ ]:
# [x] transform metal data
# Input:    
# Output:    

# (1) load original data
og_metal_data = pd.read_csv(env_data_in("metal"))

# (2) extract the required data
res_metal: Dict[str, float] = {}
for _, row in og_metal_data.iterrows():
    country = row["Entity"]
    value = row["Mismanaged plastic waste (metric tons year-1)"]
    res_metal[country] = value

# (3) transform the extracted data
max_value = max(res_metal.values())
df_metal = transform_dictionary(res_metal, lambda data: nf_log2(data, max_value))
df_metal.to_csv(env_pain_out("metal"), index=False)

df_metal.plot(kind="bar")

# Hollistic Env Pain

In [ ]:
def combine_env_pain(pain_values: List[float]) -> float:
    return 1 - max(pain_values)

In [ ]:
# ENV PAIN - this already uses SOV data!
elements = ["fire", "earth", "water", "metal", "wood"]
DFS = {}
country_set = None
for elem in elements:
    df_cur = pd.read_csv(os.path.join(BASE_PATH, "out", "sov_data", f"env-{elem}-sov.csv"))
    if country_set is None:
        country_set = set(df_cur["Country"])
    else:
        country_set = set(df_cur["Country"]).intersection(country_set)
    DFS[elem] = df_cur

env_result = []
for _, row in DFS["fire"].iterrows():
    country = row["Country"]
    if country in country_set:
        pain_vals = []
        for element in elements:
            df_cur = DFS[element]
            cur_val = df_cur.loc[df_cur["Country"] == country].iloc[0]["value"]
            pain_vals.append(cur_val)
        env_result.append({
            "Country": country,
            "value":  combine_env_pain(pain_vals)
        })

df_env_pain = pd.DataFrame(env_result)
df_env_pain.to_csv(os.path.join(BASE_PATH, "out", "hollistic_pains", "env-pain.csv"), index=False)

print(country_set)
print(len(country_set))